# Consumer Loan Credit Risk Modeling and Expected Loss Analysis

**Technical walkthrough.** Probability-of-default modelling, expected credit loss and scenario
analysis on the UCI *Default of Credit Card Clients* dataset (Taiwan, 2005).

This notebook is the readable version of the pipeline. It calls the same modules in `src/`
that `python -m src.run_pipeline` calls, so nothing here can drift away from the code that
produces the repository's published numbers.

**Contents**

1. The question, and what would count as an answer
2. Data integrity and cleaning
3. What the data says before any model
4. The leakage boundary: behavioural versus origination
5. The model ladder
6. Evaluation: discrimination, calibration, and the two errors
7. Expected loss: `EL = PD x LGD x EAD`
8. Scenario analysis
9. What drives the score
10. Limitations

## 1. The question, and what would count as an answer

**Business question.** Can borrower and account characteristics be used to estimate the
probability of default, rank accounts by risk, and estimate expected credit losses under
different scenarios?

Three things have to be true for the answer to be useful:

1. The model must **rank** accounts (ROC-AUC, Gini, KS, and a monotone decile table).
2. The probabilities must be **calibrated**, because expected loss multiplies PD *levels* -
   a model that ranks perfectly but sits 30% low produces a loss number that is 30% low.
3. The result must survive being **turned into a decision**, with the two errors priced
   differently: declining a good borrower costs forgone margin, approving a bad one costs
   `LGD x EAD`.

Accuracy is not on that list. With a 22% default rate, approving everyone is 78% accurate.

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src import data_prep, evaluate as E, expected_loss as EL, scenarios as S, explainability as X
from src.models import load_model

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("Working from:", ROOT)

Working from: /home/claude/credit-risk-el-modeling


## 2. Data integrity and cleaning

Before anything else, the loaded file is checked against the statistics published for the UCI
dataset. If a mirror has been truncated or altered, the pipeline stops here rather than
quietly training on the wrong data.

In [2]:
raw = data_prep.load_raw()
integrity = data_prep.validate_raw(raw)
integrity

{'rows': 30000,
 'columns': 25,
 'defaults': 6636,
 'default_rate': 0.2212,
 'missing_values': 0,
 'duplicate_ids': 0,
 'duplicate_rows_excl_id': 35}

30,000 accounts, 6,636 defaults, a 22.12% default rate, no missing values - matching the
published figures exactly.

Two things the raw file does contain: 35 rows that are exact duplicates once `ID` is removed,
and several undocumented category codes.

In [3]:
cleaned, clean_log = data_prep.clean(raw)
clean_log

{'rows_in': 30000,
 'education_codes_recoded_to_other': 468,
 'marriage_codes_recoded_to_other': 377,
 'duplicate_rows_dropped': 35,
 'rows_out': 29965,
 'default_rate_after_clean': 0.22125813449023862,
 'negative_bill_amounts': 3932}

**The decisions, and why:**

- **Duplicates dropped.** Identical accounts landing in both the training and test folds would
  inflate measured performance.
- **Undocumented `EDUCATION` (0, 5, 6) and `MARRIAGE` (0) codes collapsed into "Other."**
  Treating an undocumented code as its own risk level invents signal that is not there.
- **Negative bill amounts kept.** These are credit balances from overpayments, not errors.
  Utilisation features floor them at zero.
- **`PAY_0` renamed `PAY_1`** so repayment status lines up with the bill and payment columns.

One subtlety that matters more than it looks. The `PAY_*` scale runs -2 (no usage), -1 (paid in
full), 0 (revolving, minimum paid), then 1-8 months past due. It is **not monotone in risk**:
-2 and -1 are qualitatively different states, not smaller amounts of delinquency. So the raw
codes are never fed to a model as a continuous variable - they are split into non-negative
months-past-due plus explicit counts of revolving, full-payment and inactive months.

In [4]:
featured = data_prep.engineer_features(cleaned)
splits = data_prep.split_data(featured)
train, val, test = splits["train"], splits["val"], splits["test"]

pd.DataFrame({
    "rows": {k: len(v) for k, v in splits.items()},
    "defaults": {k: int(v[C.TARGET].sum()) for k, v in splits.items()},
    "default rate": {k: v[C.TARGET].mean() for k, v in splits.items()},
}).style.format({"default rate": "{:.2%}", "rows": "{:,}", "defaults": "{:,}"})

,rows,defaults,default rate
train,"17,979","3,978",22.13%
val,"5,993","1,326",22.13%
test,"5,993","1,326",22.13%


A stratified 60 / 20 / 20 split. **A time-based split is not possible here** - every account
is observed over the same April-September 2005 window with the same October outcome month, so
there is no time dimension to split on. That is a real limitation, recorded in the model card
rather than papered over.

The folds have distinct jobs:

- **train** - fit the models;
- **validation** - choose hyper-parameters, calibrate probabilities, set the decision cutoff;
- **test** - scored once, at the end. Every number reported below comes from it.

## 3. What the data says before any model

Worth looking at the portfolio directly before fitting anything, if only to know what the model
should be finding.

In [5]:
full = pd.concat(splits.values(), ignore_index=True)

delinq = full.copy()
delinq["worst_arrears"] = delinq["dpd_max"].clip(upper=5)
summary = delinq.groupby("worst_arrears").agg(
    accounts=(C.TARGET, "size"), default_rate=(C.TARGET, "mean")
)
summary["share_of_portfolio"] = summary["accounts"] / summary["accounts"].sum()
summary.style.format({"accounts": "{:,}", "default_rate": "{:.1%}", "share_of_portfolio": "{:.1%}"})

,accounts,default_rate,share_of_portfolio
worst_arrears,,,
0,"19,918",11.7%,66.5%
1,"1,668",25.1%,5.6%
2,"7,187",43.6%,24.0%
3,789,62.2%,2.6%
4,217,64.1%,0.7%
5,186,64.0%,0.6%


This single table is most of the story. Accounts that were never late in six months default
at roughly one in eight; accounts two or more cycles down default at over two in three. Any
model that does not find this is broken.

The demographic splits are much weaker by comparison.

In [6]:
for col in ["limit_band", "age_band", "education_label"]:
    g = full.groupby(col, observed=True)[C.TARGET].agg(["size", "mean"])
    g.columns = ["accounts", "default_rate"]
    print(f"\n{col}")
    print(g.assign(default_rate=lambda d: d.default_rate.map("{:.1%}".format)).to_string())


limit_band
            accounts default_rate
limit_band                       
Q1 lowest       7673        31.8%
Q2              4817        25.8%
Q3              6114        19.9%
Q4              5412        16.9%
Q5 highest      5949        13.8%

age_band
          accounts default_rate
age_band                       
21-25         3868        26.7%
26-30         7129        20.2%
31-35         5790        19.4%
36-40         4912        21.6%
41-50         5997        23.3%
51-60         1997        25.2%
60+            272        26.8%

education_label
                 accounts default_rate
education_label                       
Graduate school     10563        19.2%
High school          4915        25.2%
Other                 468         7.1%
University          14019        23.7%


Credit-limit quintile separates risk (low-limit accounts default far more), but that is
largely the issuer's own past risk assessment showing through - the limit was set by an earlier
underwriting decision. Age and education move the rate far less.

## 4. The leakage boundary: behavioural versus origination

The obvious leakage check passes trivially: the target is October 2005 and every feature is
dated September 2005 or earlier.

The harder question is **which decision** the model supports. A model with six months of
repayment history is a *behavioural* scorecard - the tool for re-underwriting, limit management
and collections on an existing account. It is not an *application* scorecard, because at
origination none of that history exists.

So both are built, and both are reported. Quoting the behavioural AUC as though it applied at
account opening would be leakage relative to the decision being made.

In [7]:
for name, fs in data_prep.FEATURE_SETS.items():
    print(f"=== {name} ({len(fs.all)} features) ===")
    print(fs.rationale)
    print("features:", ", ".join(fs.all))
    print()

=== behavioural (28 features) ===
Everything a card issuer observes on an existing account at the September 2005 cut-off: the credit line, demographics captured at opening, and six months of billing, payment and repayment-status history. This is a behavioural scorecard.
features: log_limit_bal, AGE, util_latest, util_mean, util_max, util_trend, util_volatility, months_over_90pct_util, pay_ratio_latest, pay_ratio_mean, pay_ratio_min, paid_to_billed_6m, months_paid_nothing, dpd_latest, dpd_max, months_delinquent, months_2plus_delinquent, dpd_trend, revolver_months, full_payer_months, inactive_months, bill_latest, bill_mean, bill_growth_6m, available_credit, sex_label, education_label, marriage_label

=== origination (5 features) ===
Only what is known when the account is opened: the assigned credit line and application demographics. No repayment history exists at that point, so this is the fair comparison for an application scorecard and shows how much of the behavioural model's power co

In [8]:
fs_comparison = pd.read_csv(C.TABLES / "feature_set_comparison.csv")
fs_comparison.style.format({c: "{:.4f}" for c in ["roc_auc", "gini", "pr_auc", "ks", "brier"]})

,feature_set,model,n_features,roc_auc,gini,pr_auc,ks,brier
0,behavioural,Logistic regression,28,0.7643,0.5286,0.5168,0.4089,0.1381
1,behavioural,Gradient boosting (XGBoost),28,0.7779,0.5557,0.5516,0.4271,0.1351
2,origination,Logistic regression,5,0.6217,0.2434,0.2969,0.1841,0.1671
3,origination,Gradient boosting (XGBoost),5,0.6267,0.2533,0.3153,0.2003,0.1664


Removing the repayment history costs roughly 0.15 of ROC-AUC. The origination-only model is
only modestly better than chance, and that gap is the honest measure of how much of this
model's power an application-time user would **not** get.

## 5. The model ladder

Five families, each fitted with a small explicit grid and selected on the validation fold:

1. **Baseline** - predicts the training default rate for everyone. No discriminatory power by
   construction; it exists to define zero.
2. **Logistic regression** - the scorecard reference. Monotone in each input, coefficients
   readable as log-odds, defensible in a credit committee.
3. **Decision tree** - splits read as policy rules.
4. **Random forest** - shows how much of the gain is ensembling rather than boosting.
5. **Gradient boosting** (LightGBM and XGBoost) - captures interactions the scorecard cannot.

Every model above the logistic regression has to justify its complexity on discrimination
*and* calibration.

In [9]:
# Models are fitted by `python -m src.models` / `python -m src.run_pipeline`; loaded here.
import json
manifest = json.loads((C.MODELS / "manifest_behavioural.json").read_text())
pd.DataFrame([
    {"model": k, "validation ROC-AUC": v["val_auc_uncalibrated"],
     "grid points searched": v["n_grid_points"], "chosen parameters": v["params"]}
    for k, v in manifest["models"].items()
])

,model,validation ROC-AUC,grid points searched,chosen parameters
0,Baseline (prior),0.500000,1,{}
1,Logistic regression,0.777717,4,{'C': 0.1}
2,Decision tree,0.779497,8,"{'max_depth': 6, 'min_samples_leaf': 200}"
3,Random forest,0.790218,4,"{'n_estimators': 400, 'max_depth': 8, 'min_sam..."
4,Gradient boosting (LightGBM),0.791757,8,"{'n_estimators': 300, 'learning_rate': 0.03, '..."
5,Gradient boosting (XGBoost),0.792683,8,"{'n_estimators': 300, 'learning_rate': 0.03, '..."


### Why calibration is a separate step

Discrimination and calibration are different properties. A model can rank every account
correctly and still be systematically wrong about the *level* of risk - and expected loss
multiplies levels, not ranks.

Probabilities are therefore calibrated on the validation fold, with the base model frozen so
calibration never refits on training data. Isotonic and Platt scaling are compared by five-fold
cross-validation *inside* the validation fold - judging a calibrator on the rows it was fitted
on would flatter the more flexible method - and the winner is refitted on the whole fold.

The calibrated probability is then floored and capped. Isotonic calibration is a step function
and will hand out a PD of exactly 0 or exactly 1 in its outermost steps; a PD of zero asserts
that no loss is possible and would zero out that account's expected loss. Supervisory
frameworks use an explicit floor for exactly this reason - Basel sets 0.03% for retail
exposures - and the same device is applied here.

In [10]:
fs = data_prep.BEHAVIOURAL
feats = fs.all
champion = json.loads((C.TABLES / "results.json").read_text())["champion_model"]

cal_model = load_model("behavioural", champion, calibrated=True)
raw_model = load_model("behavioural", champion, calibrated=False)
logit_model = load_model("behavioural", "Logistic regression", calibrated=True)

y_test = test[C.TARGET].to_numpy()
p_test = cal_model.predict_proba(test[feats])[:, 1]
p_test_raw = raw_model.predict_proba(test[feats])[:, 1]
p_logit = logit_model.predict_proba(test[feats])[:, 1]

print(f"Champion: {champion}")
print(f"mean predicted PD (calibrated):   {p_test.mean():.4f}")
print(f"mean predicted PD (uncalibrated): {p_test_raw.mean():.4f}")
print(f"observed default rate:            {y_test.mean():.4f}")

Champion: Gradient boosting (XGBoost)
mean predicted PD (calibrated):   0.2255
mean predicted PD (uncalibrated): 0.2199
observed default rate:            0.2213


## 6. Evaluation

### 6.1 Discrimination and calibration

In [11]:
comparison = pd.read_csv(C.TABLES / "model_comparison.csv")
comparison[["model", "val_auc", "roc_auc", "pr_auc", "ks", "brier", "ece"]].style.format(
    {c: "{:.4f}" for c in ["val_auc", "roc_auc", "pr_auc", "ks", "brier", "ece"]}
)

,model,val_auc,roc_auc,pr_auc,ks,brier,ece
0,Baseline (prior),0.5000,0.5000,0.2213,0.0000,0.1723,0.0000
1,Logistic regression,0.7812,0.7643,0.5168,0.4089,0.1381,0.0117
2,Decision tree,0.7808,0.7671,0.5147,0.4080,0.1375,0.0161
3,Random forest,0.7937,0.7752,0.5349,0.4120,0.1354,0.0171
4,Gradient boosting (LightGBM),0.7918,0.7780,0.5664,0.4296,0.1351,0.0136
5,Gradient boosting (XGBoost),0.7958,0.7779,0.5516,0.4271,0.1351,0.0174


Reading this table:

- **ROC-AUC / Gini / KS** - can the model tell good from bad? Boosting wins, but not by much.
- **PR-AUC** - performance on the minority class, against a 22% base rate. This is the honest
  view for a rare event, and the gap between models is wider here than on ROC-AUC.
- **Brier / calibration error** - are the probabilities usable as levels?

**The gradient-boosting gain over the scorecard is real but small.** On this dataset a fully
interpretable logistic regression gets most of the way, and that is worth stating plainly
rather than burying: the complexity of a boosted model has to be justified by more than a
third decimal place.

In [12]:
stability = X.stability_check(cal_model, splits, feats)
psi = X.population_stability_index(cal_model.predict_proba(train[feats])[:, 1], p_test)
print(stability.to_string(index=False))
print(f"\nPSI (train vs test score distribution): {psi:.4f}")

split  accounts  roc_auc
train     17979 0.811462
  val      5993 0.795840
 test      5993 0.777860

PSI (train vs test score distribution): 0.0007


Train ROC-AUC exceeds test by about 0.03 - mild optimism, not collapse. The PSI is
essentially zero, which is expected for a random split of a single cohort and is *not* evidence
of stability through time.

In [13]:
cal_table = E.calibration_table(y_test, p_test)
cal_table.style.format({"predicted": "{:.2%}", "observed": "{:.2%}", "gap": "{:+.2%}"})

,bin,accounts,predicted,observed,gap
0,1,600,3.14%,4.17%,-1.03%
1,2,599,6.00%,7.35%,-1.34%
2,3,599,8.49%,9.02%,-0.53%
3,4,599,10.03%,11.02%,-0.99%
4,5,600,14.67%,16.50%,-1.83%
5,6,599,18.23%,15.86%,+2.37%
6,7,599,21.10%,18.36%,+2.73%
7,8,599,26.67%,27.88%,-1.21%
8,9,599,44.37%,41.40%,+2.97%
9,10,600,72.72%,69.67%,+3.05%


### 6.2 Rank-ordering

The decile table is the form a credit committee reads.

In [14]:
ead_test = EL.account_level_el(test, p_test)["ead"].to_numpy()
deciles = E.decile_table(y_test, p_test, exposure=ead_test)
deciles.style.format({
    "observed_default_rate": "{:.1%}", "mean_predicted_pd": "{:.1%}",
    "min_predicted_pd": "{:.1%}", "max_predicted_pd": "{:.1%}",
    "lift": "{:.2f}x", "exposure": "{:,.0f}", "share_of_exposure": "{:.1%}",
    "cumulative_defaults_captured": "{:.0%}", "cumulative_accounts": "{:.0%}",
})

,decile,accounts,defaults,observed_default_rate,mean_predicted_pd,min_predicted_pd,max_predicted_pd,exposure,share_of_exposure,lift,cumulative_defaults_captured,cumulative_accounts
0,1,600,25,4.2%,3.1%,0.0%,5.1%,"90,106,109",16.4%,0.19x,100%,100%
1,2,599,44,7.3%,6.0%,5.1%,7.7%,"77,860,401",14.2%,0.33x,98%,90%
2,3,599,54,9.0%,8.5%,7.7%,8.6%,"64,635,478",11.8%,0.41x,95%,80%
3,4,599,66,11.0%,10.0%,8.6%,10.8%,"59,299,516",10.8%,0.50x,91%,70%
4,5,600,99,16.5%,14.7%,10.8%,16.7%,"52,118,861",9.5%,0.75x,86%,60%
5,6,599,95,15.9%,18.2%,16.7%,20.0%,"44,300,017",8.1%,0.72x,78%,50%
6,7,599,110,18.4%,21.1%,20.0%,22.3%,"41,527,251",7.6%,0.83x,71%,40%
7,8,599,167,27.9%,26.7%,22.3%,38.3%,"38,908,189",7.1%,1.26x,63%,30%
8,9,599,248,41.4%,44.4%,38.3%,58.1%,"37,010,773",6.7%,1.87x,50%,20%
9,10,600,418,69.7%,72.7%,58.1%,100.0%,"43,240,602",7.9%,3.15x,32%,10%


### 6.3 The two errors, and where to stand

`predict()` returns a 0.5 cutoff. That is a statistical convention, not a credit policy, and on
this portfolio it is the wrong place to stand:

- a **false negative** (approve, then default) costs `LGD x EAD`;
- a **false positive** (decline someone who would have paid) costs the forgone margin on the
  balance that customer would have carried.

Under the working assumptions those differ by roughly an order of magnitude.

In [15]:
results = json.loads((C.TABLES / "results.json").read_text())
th = results["thresholds"]
cutoffs = pd.read_csv(C.TABLES / "cutoff_comparison.csv")
cutoffs.style.format({
    "threshold": "{:.1%}", "flagged_rate": "{:.1%}", "precision": "{:.3f}",
    "recall": "{:.3f}", "f1": "{:.3f}", "specificity": "{:.3f}", "accuracy": "{:.3f}",
})

,cutoff_rule,threshold,flagged_rate,precision,recall,f1,specificity,accuracy,tp,fp,tn,fn
0,risk_appetite_15%_bad_rate,44.7%,15.4%,0.618,0.430,0.507,0.925,0.815,570.000000,352.000000,4315.000000,756.000000
1,f1_optimal,27.5%,25.4%,0.501,0.575,0.535,0.837,0.779,762.000000,759.000000,3908.000000,564.000000
2,cost_minimising,5.1%,92.1%,0.237,0.986,0.382,0.098,0.294,1307.000000,4211.000000,456.000000,19.000000
3,naive_0.5,50.0%,12.4%,0.663,0.373,0.477,0.946,0.819,494.000000,251.000000,4416.000000,832.000000


Four candidate rules, and they disagree - which is the point.

The **cost-minimising** cutoff comes out extremely tight. That is not a discovery about credit;
it is an artefact of weighing a one-period loss against one period of margin. The sensitivity
table below shows the implied cutoff moving a long way as the assumed margin changes, which is
precisely why a cutoff should not be read off a cost calculation alone.

In [16]:
pd.read_csv(C.TABLES / "cutoff_margin_sensitivity.csv").style.format({
    "assumed_margin_on_good_account": "{:.0%}", "implied_cutoff": "{:.1%}",
    "approval_rate": "{:.1%}", "bad_rate_of_approved": "{:.2%}", "net_cost": "{:,.0f}",
})

,assumed_margin_on_good_account,implied_cutoff,approval_rate,bad_rate_of_approved,net_cost
0,6%,5.1%,8.2%,2.64%,"13,628,839"
1,10%,7.7%,18.7%,4.19%,"20,781,874"
2,15%,16.7%,44.9%,7.25%,"25,869,786"
3,20%,20.0%,56.1%,9.16%,"29,912,157"
4,30%,20.0%,56.1%,9.16%,"35,300,158"
5,45%,23.2%,72.2%,11.78%,"39,828,256"
6,60%,23.2%,72.2%,11.78%,"43,862,138"


So the deployed cutoff is set the way credit policy is actually set: risk appetite is stated
as a tolerable default rate on the approved book, and the cutoff is whatever delivers it.

In [17]:
policy_cutoff = th["policy_cutoff_from_validation"]
el_base = EL.account_level_el(test, p_test, lgd=C.LGD_BASELINE, ccf=C.CCF_BASELINE)
decision = EL.decision_rule_evaluation(el_base, policy_cutoff)
pd.Series(decision).to_frame("value")

,value
threshold,4.470588e-01
approval_rate,8.461538e-01
decline_rate,1.538462e-01
accounts_declined,9.220000e+02
bad_rate_of_approved,1.490830e-01
bad_rate_of_declined,6.182213e-01
bad_rate_no_cutoff,2.212581e-01
defaults_avoided,5.700000e+02
defaults_retained,7.560000e+02
good_accounts_declined,3.520000e+02


**What this is, and is not.** It is a retrospective evaluation on held-out accounts: applying
this cutoff to this population would have cut the approved book's realised default rate from
22.1% to about 15%, while declining roughly 350 accounts that in fact paid.

It is **not** a claim that lending decisions were improved. No live decisions were made, and
the accounts the rule declines were extended credit in reality - so their behaviour under a
decline is unobservable.

## 7. Expected loss

$$EL = PD \times LGD \times EAD$$

| Component | Source | Status |
| --- | --- | --- |
| **PD** | Calibrated model | **Estimated** |
| **EAD** | Drawn balance + CCF x undrawn line | Balance observed, **CCF assumed** |
| **LGD** | Config constant | **Assumed** - the dataset has no recovery data |

**EAD is not just today's balance.** A card is a revolving commitment; distressed borrowers
typically draw further before they stop paying. The credit conversion factor is the assumed
share of the undrawn line drawn between now and default. Using the balance alone understates
exposure, using the full line overstates it.

In [18]:
summary = EL.portfolio_summary(el_base)
pd.Series(summary).to_frame("value")

,value
accounts,5.993000e+03
total_limit,1.007540e+09
total_drawn_balance,3.077440e+08
total_ead,5.490072e+08
total_expected_loss,6.567132e+07
el_rate_on_ead,1.196183e-01
average_pd,2.254533e-01
exposure_weighted_pd,1.840282e-01
lgd_assumed,6.500000e-01
ccf_assumed,3.500000e-01


### A partial check on the loss number

The PD and EAD components can be tested against outcomes, even though LGD cannot. Holding the
LGD assumption fixed, the loss implied by the accounts that actually defaulted should be close
to the modelled expected loss.

In [19]:
modelled = summary["total_expected_loss"]
realised = summary["realised_loss_at_assumed_lgd"]
print(f"Modelled expected loss:                 {C.CURRENCY}{modelled:,.0f}")
print(f"Loss implied by actual defaults (same LGD): {C.CURRENCY}{realised:,.0f}")
print(f"Gap: {modelled / realised - 1:+.2%}")

Modelled expected loss:                 NT$65,671,322
Loss implied by actual defaults (same LGD): NT$67,487,513
Gap: -2.69%


Close. That validates PD and EAD together. It says **nothing** about whether the 65% LGD
assumption is right - only that if it were right, the loss estimate would be about right.

In [20]:
by_band = EL.el_by_risk_band(el_base)
by_band.style.format({
    "mean_pd": "{:.2%}", "observed_default_rate": "{:.2%}", "total_ead": "{:,.0f}",
    "expected_loss": "{:,.0f}", "el_rate_on_ead": "{:.2%}", "share_of_ead": "{:.1%}",
    "share_of_el": "{:.1%}", "el_concentration": "{:.2f}x",
})

,risk_decile,accounts,mean_pd,total_ead,expected_loss,observed_default_rate,el_rate_on_ead,share_of_ead,share_of_el,el_concentration
0,1,600,3.14%,"90,106,109","1,785,488",4.17%,1.98%,16.4%,2.7%,0.17x
1,2,599,6.00%,"77,860,401","3,015,531",7.35%,3.87%,14.2%,4.6%,0.32x
2,3,599,8.49%,"64,635,478","3,560,006",9.02%,5.51%,11.8%,5.4%,0.46x
3,4,599,10.03%,"59,299,516","3,844,224",11.02%,6.48%,10.8%,5.9%,0.54x
4,5,600,14.67%,"52,118,861","4,931,586",16.50%,9.46%,9.5%,7.5%,0.79x
5,6,599,18.23%,"44,300,017","5,236,697",15.86%,11.82%,8.1%,8.0%,0.99x
6,7,599,21.10%,"41,527,251","5,674,112",18.36%,13.66%,7.6%,8.6%,1.14x
7,8,599,26.67%,"38,908,189","6,681,363",27.88%,17.17%,7.1%,10.2%,1.44x
8,9,599,44.37%,"37,010,773","10,635,210",41.40%,28.74%,6.7%,16.2%,2.40x
9,10,600,72.72%,"43,240,602","20,307,104",69.67%,46.96%,7.9%,30.9%,3.93x


In [21]:
EL.el_by_group(el_base, "limit_band").style.format({
    "mean_pd": "{:.2%}", "observed_default_rate": "{:.2%}", "total_ead": "{:,.0f}",
    "expected_loss": "{:,.0f}", "el_rate_on_ead": "{:.2%}", "share_of_ead": "{:.1%}",
    "share_of_el": "{:.1%}", "el_concentration": "{:.2f}x",
})

,limit_band,accounts,mean_pd,total_ead,expected_loss,observed_default_rate,el_rate_on_ead,share_of_ead,share_of_el,el_concentration
0,Q1 lowest,1510,33.45%,"40,190,150","8,206,142",31.06%,20.42%,7.3%,12.5%,1.71x
1,Q2,965,25.84%,"53,349,929","9,031,199",27.25%,16.93%,9.7%,13.8%,1.42x
2,Q3,1240,20.34%,"107,191,331","14,369,281",20.24%,13.41%,19.5%,21.9%,1.12x
3,Q4,1069,16.37%,"124,082,458","13,399,583",14.87%,10.80%,22.6%,20.4%,0.90x
4,Q5 highest,1209,14.01%,"224,193,328","20,665,118",15.22%,9.22%,40.8%,31.5%,0.77x


## 8. Scenario analysis

Three scenarios, each moving PD, LGD and the credit conversion factor together - in a downturn
they do not move independently.

**How the PD stress works.** The multiplier acts on the *odds* of default, not the probability:

$$\text{odds}' = m \times \text{odds}, \qquad PD' = \frac{\text{odds}'}{1 + \text{odds}'}$$

This is equivalent to shifting every score by a constant $\log(m)$ on the log-odds scale. A
flat multiplier on the probability itself would push high-risk accounts above 1.0 and would
stress a 1% account and a 40% account by wildly different amounts in score terms.

In [22]:
for p0 in (0.01, 0.10, 0.40, 0.80):
    print(f"PD {p0:>5.0%} -> x1.5 odds -> {S.stress_pd(np.array([p0]), 1.5)[0]:.3f}"
          f"   |  x2.25 odds -> {S.stress_pd(np.array([p0]), 2.25)[0]:.3f}")

PD    1% -> x1.5 odds -> 0.015   |  x2.25 odds -> 0.022
PD   10% -> x1.5 odds -> 0.143   |  x2.25 odds -> 0.200
PD   40% -> x1.5 odds -> 0.500   |  x2.25 odds -> 0.600
PD   80% -> x1.5 odds -> 0.857   |  x2.25 odds -> 0.900


In [23]:
scen_table, scen_frames = S.run_all_scenarios(test, p_test)
scen_table[["scenario", "pd_odds_multiplier", "lgd_assumed", "ccf_assumed", "average_pd",
            "total_ead", "total_expected_loss", "el_rate_on_ead", "el_vs_baseline_pct"]].style.format({
    "pd_odds_multiplier": "{:.2f}x", "lgd_assumed": "{:.0%}", "ccf_assumed": "{:.0%}",
    "average_pd": "{:.2%}", "total_ead": "{:,.0f}", "total_expected_loss": "{:,.0f}",
    "el_rate_on_ead": "{:.2%}", "el_vs_baseline_pct": "{:+.0%}",
})

,scenario,pd_odds_multiplier,lgd_assumed,ccf_assumed,average_pd,total_ead,total_expected_loss,el_rate_on_ead,el_vs_baseline_pct
0,Baseline,1.00x,65%,35%,22.55%,"549,007,196","65,671,322",11.96%,+0%
1,Moderate deterioration,1.50x,75%,45%,28.33%,"619,550,704","107,515,246",17.35%,+64%
2,Severe deterioration,2.25x,85%,55%,34.95%,"690,094,213","168,871,865",24.47%,+157%


### How much of the answer is the assumption?

Because LGD is assumed rather than estimated, the honest presentation of expected loss is a
surface, not a point.

In [24]:
grid = S.sensitivity_grid(test, p_test)
pivot = grid.pivot(index="lgd", columns="pd_odds_multiplier", values="total_expected_loss") / 1e6
pivot.style.format("{:,.0f}").background_gradient(cmap="Blues")

pd_odds_multiplier,1.000000,1.250000,1.500000,1.750000,2.000000,2.250000,2.500000
lgd,,,,,,,
0.450000,45,52,58,63,68,72,77
0.550000,56,64,71,77,83,89,94
0.650000,66,75,84,91,98,105,111
0.750000,76,87,96,105,113,121,128
0.850000,86,98,109,119,128,137,145


Read across a row and the movement is the model's stress. Read down a column and it is the
LGD assumption alone. They are comparable in size - which is exactly why the assumption is
labelled everywhere it appears, and why sourcing real recovery data is the single
highest-value next step for this analysis.

## 9. What drives the score

Three views, because one importance ranking is not an explanation.

In [25]:
odds = X.logistic_odds_ratios(load_model("behavioural", "Logistic regression", calibrated=False))
odds.head(12).style.format({"coefficient_log_odds": "{:+.3f}", "odds_ratio": "{:.3f}"})

,feature,coefficient_log_odds,odds_ratio,direction
0,months_2plus_delinquent,+0.699,2.012,increases risk
1,education_label_Other,-0.604,0.547,reduces risk
2,util_max,-0.391,0.677,reduces risk
3,months_delinquent,-0.370,0.691,reduces risk
4,dpd_latest,+0.355,1.426,increases risk
5,util_mean,+0.308,1.361,increases risk
6,log_limit_bal,-0.292,0.746,reduces risk
7,full_payer_months,+0.270,1.310,increases risk
8,bill_latest,+0.246,1.279,increases risk
9,pay_ratio_mean,-0.227,0.797,reduces risk


Coefficients are on standardised inputs, so each is the change in log-odds per one standard
deviation of that feature (or versus the dropped reference level for the one-hot columns). An
odds ratio above 1 raises the odds of default.

Note the correlated-feature effect: several delinquency measures carry overlapping information,
so individual coefficients trade off against each other and should be read as a group rather
than one at a time. This is a normal scorecard problem, and it is one reason the permutation
and SHAP views below are worth having.

In [26]:
perm = pd.read_csv(C.TABLES / "permutation_importance.csv")
shap_imp = pd.read_csv(C.TABLES / "shap_global_importance.csv")
pd.concat([
    perm.head(10)[["feature", "auc_drop_mean"]].reset_index(drop=True),
    shap_imp.head(10)[["feature", "mean_abs_shap"]].reset_index(drop=True),
], axis=1)

,feature,auc_drop_mean,feature,mean_abs_shap
0,dpd_latest,0.032705,dpd_latest,0.353849
1,dpd_max,0.014088,dpd_max,0.277328
2,available_credit,0.013561,available_credit,0.139920
3,bill_mean,0.008006,bill_mean,0.131946
4,bill_latest,0.007330,bill_latest,0.131358
5,months_2plus_delinquent,0.002564,months_2plus_delinquent,0.110082
6,log_limit_bal,0.002513,log_limit_bal,0.077191
7,education_label,0.001802,pay_ratio_min,0.070534
8,inactive_months,0.001761,months_delinquent,0.062959
9,paid_to_billed_6m,0.001284,pay_ratio_mean,0.058972


All three views agree: recent arrears status dominates, then the depth and persistence of
delinquency, then available credit and balance level. Demographics sit far down the list.

The clean read for a risk audience: **how the customer has been paying matters far more than
who the customer is.**

In [27]:
# Account-level reason codes - the adverse-action-style view of one account's score.
pd.read_csv(C.TABLES / "reason_codes_high_risk_account.csv").style.format({
    "feature_value": "{:.3f}", "shap_value": "{:+.3f}"
})

,feature,feature_value,shap_value,effect
0,dpd_latest,3.000,+1.004,raises PD
1,dpd_max,7.000,+0.456,raises PD
2,months_2plus_delinquent,6.000,+0.322,raises PD
3,log_limit_bal,9.210,+0.157,raises PD
4,bill_latest,2300.000,+0.151,raises PD


## 10. Limitations

1. **One cohort, one outcome month.** No out-of-time validation; the split is
   stratified-random, not time-based.
2. **Taiwan, 2005.** The portfolio follows a domestic card-debt crisis. A 22% default rate is
   not a through-the-cycle rate and none of these levels should be read across to another book.
3. **Behavioural, not application.** The champion model needs six months of repayment history.
4. **LGD is assumed.** Every currency figure inherits that assumption one-for-one.
5. **No macroeconomic variables.** Unemployment and rates enter only as scenario multipliers.
6. **Demographic inputs.** Sex, education and marital status are present in the data and are
   used here for segment monitoring. Several are prohibited or restricted inputs for credit
   decisions in many jurisdictions; a deployable model would exclude them and be tested for
   disparate impact.

### Next steps

- Source recovery and collection data to replace the assumed LGD with an estimated one.
- Re-fit on multiple cohorts to enable out-of-time validation and a real PSI baseline.
- Rebuild without demographic inputs and quantify what discrimination that costs.
- Extend to a lifetime PD term structure if the goal is IFRS 9 / CECL-style reporting.

---

*Portfolio project built on public data. Not investment, credit or financial advice.
Reproduce end to end with `python -m src.run_pipeline`.*